# Baseline Forecasting and Rolling-Origin Validation

This notebook establishes simple, leakage-safe forecasting benchmarks for the
selected M5 product-store series.

The forecasting horizon is 28 days. Model performance will be measured using
multiple historical validation windows within `d_1` to `d_1913`.

The protected evaluation period, `d_1914` to `d_1941`, will not be accessed
during model development.

## Objectives

1. Create 28-day rolling-origin validation folds.
2. Implement simple forecasting baselines.
3. Evaluate forecasts using MAE, RMSE, WAPE, and bias.
4. Compare performance by store, category, demand pattern, and horizon.
5. Establish the benchmark that future machine-learning models must beat.

## 1. Setup and Data Loading

We begin by importing the required libraries and loading the canonical sales
dataset.

The notebook performs several checks to ensure:

- only development data up to `d_1913` are present;
- dates are stored as datetime values;
- every product-store series is ordered chronologically;
- the protected 28-day holdout has not leaked into development.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.3f}".format)

FORECAST_HORIZON = 28
DEVELOPMENT_END_DAY = 1913
PROTECTED_HOLDOUT_START = 1914
PROTECTED_HOLDOUT_END = 1941

SERIES_KEYS = [
    "store_id",
    "product_id",
]

TARGET_COLUMN = "units_sold"

In [19]:
from pathlib import Path

# Search the current directory and its parents for the project root.
current_directory = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [
    current_directory,
    *current_directory.parents,
]:
    expected_dataset = (
        candidate
        / "data"
        / "processed"
        / "canonical_sales.parquet"
    )

    if expected_dataset.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "The expected file is "
        "'data/processed/canonical_sales.parquet'."
    )

canonical_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "canonical_sales.parquet"
)

print("Project root:", PROJECT_ROOT)
print("Canonical dataset:", canonical_path)

Project root: C:\Users\USER\Desktop\agentic ai\ai_sales_forecasting
Canonical dataset: C:\Users\USER\Desktop\agentic ai\ai_sales_forecasting\data\processed\canonical_sales.parquet


In [ ]:
# df = pd.read_parquet(canonical_path)

# # Remove accidental whitespace from column names.
# df.columns = df.columns.astype(str).str.strip()

# print("Available columns:")
# print(df.columns.tolist())

# required_columns = {
#     "store_id",
#     "product_id",
#     "day_id",
#     "date",
#     "units_sold",
# }

# missing_columns = required_columns.difference(df.columns)

# if missing_columns:
#     raise KeyError(
#         f"Required columns are missing: {sorted(missing_columns)}"
#     )

# # Create day_num when it is not already available.
# if "day_num" not in df.columns:
#     extracted_day_number = (
#         df["day_id"]
#         .astype(str)
#         .str.extract(r"(\d+)$", expand=False)
#     )

#     if extracted_day_number.isna().any():
#         invalid_values = (
#             df.loc[
#                 extracted_day_number.isna(),
#                 "day_id",
#             ]
#             .drop_duplicates()
#             .head(10)
#             .tolist()
#         )

#         raise ValueError(
#             "Could not extract day numbers from day_id. "
#             f"Example invalid values: {invalid_values}"
#         )

#     df["day_num"] = extracted_day_number.astype("int64")

# df["date"] = pd.to_datetime(
#     df["date"],
#     errors="raise",
# )

# df = (
#     df.loc[
#         df["day_num"] <= DEVELOPMENT_END_DAY
#     ]
#     .sort_values(
#         SERIES_KEYS + ["day_num"]
#     )
#     .reset_index(drop=True)
# )

# print("\nDataset loaded successfully.")
# print("Number of rows:", f"{len(df):,}")
# print(
#     "Number of series:",
#     df[SERIES_KEYS].drop_duplicates().shape[0],
# )
# print("First day:", df["day_num"].min())
# print("Last development day:", df["day_num"].max())
# print("Start date:", df["date"].min().date())
# print("End date:", df["date"].max().date())

Available columns:
['product_id', 'department_id', 'category_id', 'store_id', 'state_id', 'day_id', 'units_sold', 'date', 'week_id', 'sell_price']

Dataset loaded successfully.
Number of rows: 58,327,370
Number of series: 30490
First day: 1
Last development day: 1913
Start date: 2011-01-29
End date: 2016-04-24


In [22]:
selected_products = subset_config["product_ids"]
selected_stores = subset_config["store_ids"]

df = (
    df.loc[
        df["product_id"].isin(selected_products)
        & df["store_id"].isin(selected_stores)
        & (df["day_num"] <= DEVELOPMENT_END_DAY)
    ]
    .sort_values(
        SERIES_KEYS + ["day_num"]
    )
    .reset_index(drop=True)
    .copy()
)

print("Subset applied successfully.")
print("Number of rows:", f"{len(df):,}")
print("Number of stores:", df["store_id"].nunique())
print("Number of products:", df["product_id"].nunique())
print(
    "Number of product-store series:",
    df[SERIES_KEYS].drop_duplicates().shape[0],
)
print("First day:", df["day_num"].min())
print("Last day:", df["day_num"].max())

Subset applied successfully.
Number of rows: 179,822
Number of stores: 2
Number of products: 47
Number of product-store series: 94
First day: 1
Last day: 1913


## 2. Rolling-Origin Validation

A random train-test split is inappropriate for time-series forecasting because
it can allow future observations to influence predictions of the past.

Instead, we use rolling-origin validation. For each fold:

1. Training data contain only observations before the cutoff.
2. The following 28 days form the validation period.
3. The training window expands as time moves forward.
4. Each forecast simulates a real 28-day forecasting exercise.

Five non-overlapping validation folds provide 140 validation days while keeping
the protected `d_1914–d_1941` period untouched.

In [23]:
N_VALIDATION_FOLDS = 5

validation_folds = []

for fold_number in range(1, N_VALIDATION_FOLDS + 1):

    # Fold 5 is the most recent fold ending on d_1913.
    periods_after_fold = (
        N_VALIDATION_FOLDS - fold_number
    )

    validation_end_day = (
        DEVELOPMENT_END_DAY
        - periods_after_fold * FORECAST_HORIZON
    )

    validation_start_day = (
        validation_end_day
        - FORECAST_HORIZON
        + 1
    )

    training_end_day = validation_start_day - 1

    validation_folds.append(
        {
            "fold": fold_number,
            "training_end_day": training_end_day,
            "validation_start_day": validation_start_day,
            "validation_end_day": validation_end_day,
        }
    )

validation_folds = pd.DataFrame(validation_folds)

display(validation_folds)

,fold,training_end_day,validation_start_day,validation_end_day
0,1,1773,1774,1801
1,2,1801,1802,1829
2,3,1829,1830,1857
3,4,1857,1858,1885
4,5,1885,1886,1913


In [24]:
day_date_lookup = (
    df[
        [
            "day_num",
            "date",
        ]
    ]
    .drop_duplicates()
    .set_index("day_num")["date"]
)

validation_folds["training_end_date"] = (
    validation_folds["training_end_day"]
    .map(day_date_lookup)
)

validation_folds["validation_start_date"] = (
    validation_folds["validation_start_day"]
    .map(day_date_lookup)
)

validation_folds["validation_end_date"] = (
    validation_folds["validation_end_day"]
    .map(day_date_lookup)
)

validation_folds = validation_folds[
    [
        "fold",
        "training_end_day",
        "training_end_date",
        "validation_start_day",
        "validation_start_date",
        "validation_end_day",
        "validation_end_date",
    ]
]

display(validation_folds)

,fold,training_end_day,training_end_date,validation_start_day,validation_start_date,validation_end_day,validation_end_date
0,1,1773,2015-12-06,1774,2015-12-07,1801,2016-01-03
1,2,1801,2016-01-03,1802,2016-01-04,1829,2016-01-31
2,3,1829,2016-01-31,1830,2016-02-01,1857,2016-02-28
3,4,1857,2016-02-28,1858,2016-02-29,1885,2016-03-27
4,5,1885,2016-03-27,1886,2016-03-28,1913,2016-04-24


In [25]:
expected_series = (
    df[SERIES_KEYS]
    .drop_duplicates()
    .shape[0]
)

expected_validation_rows = (
    expected_series
    * FORECAST_HORIZON
)

print("Expected number of series:", expected_series)
print(
    "Expected validation rows per fold:",
    f"{expected_validation_rows:,}",
)

Expected number of series: 94
Expected validation rows per fold: 2,632


In [26]:


fold_checks = []

for fold in validation_folds.itertuples(index=False):
    validation_data = df.loc[
        df["day_num"].between(
            fold.validation_start_day,
            fold.validation_end_day,
        )
    ]

    observations_per_series = (
        validation_data
        .groupby(SERIES_KEYS)
        .size()
    )

    fold_checks.append(
        {
            "fold": fold.fold,
            "validation_rows": len(validation_data),
            "number_of_series": len(
                observations_per_series
            ),
            "minimum_days_per_series": (
                observations_per_series.min()
            ),
            "maximum_days_per_series": (
                observations_per_series.max()
            ),
            "check_passed": (
                len(validation_data)
                == expected_validation_rows
                and len(observations_per_series)
                == expected_series
                and observations_per_series.min()
                == FORECAST_HORIZON
                and observations_per_series.max()
                == FORECAST_HORIZON
            ),
        }
    )

fold_checks = pd.DataFrame(fold_checks)

display(fold_checks)

if not fold_checks["check_passed"].all():
    raise ValueError(
        "One or more validation folds are incomplete."
    )

print("All rolling-origin validation folds passed.")

,fold,validation_rows,number_of_series,minimum_days_per_series,maximum_days_per_series,check_passed
0,1,2632,94,28,28,True
1,2,2632,94,28,28,True
2,3,2632,94,28,28,True
3,4,2632,94,28,28,True
4,5,2632,94,28,28,True


All rolling-origin validation folds passed.


In [27]:


fold_checks = []

for fold in validation_folds.itertuples(index=False):
    validation_data = df.loc[
        df["day_num"].between(
            fold.validation_start_day,
            fold.validation_end_day,
        )
    ]

    observations_per_series = (
        validation_data
        .groupby(SERIES_KEYS)
        .size()
    )

    fold_checks.append(
        {
            "fold": fold.fold,
            "validation_rows": len(validation_data),
            "number_of_series": len(
                observations_per_series
            ),
            "minimum_days_per_series": (
                observations_per_series.min()
            ),
            "maximum_days_per_series": (
                observations_per_series.max()
            ),
            "check_passed": (
                len(validation_data)
                == expected_validation_rows
                and len(observations_per_series)
                == expected_series
                and observations_per_series.min()
                == FORECAST_HORIZON
                and observations_per_series.max()
                == FORECAST_HORIZON
            ),
        }
    )

fold_checks = pd.DataFrame(fold_checks)

display(fold_checks)

if not fold_checks["check_passed"].all():
    raise ValueError(
        "One or more validation folds are incomplete."
    )

print("All rolling-origin validation folds passed.")

,fold,validation_rows,number_of_series,minimum_days_per_series,maximum_days_per_series,check_passed
0,1,2632,94,28,28,True
1,2,2632,94,28,28,True
2,3,2632,94,28,28,True
3,4,2632,94,28,28,True
4,5,2632,94,28,28,True


All rolling-origin validation folds passed.


## 3. Baseline 1: 28-Day Seasonal Naïve Forecast

The first benchmark predicts demand using sales from exactly 28 days earlier:

$$
\hat{y}_{t} = y_{t-28}
$$

For example, the forecast for `d_1886` uses the observed demand from `d_1858`.

This baseline is useful because:

- 28 days equal exactly four weeks;
- the forecast uses the same day of the week;
- every prediction uses information available before the forecast begins;
- it provides a strong but simple benchmark for weekly retail demand.

A future model should outperform this baseline consistently, not merely on one
validation period.

In [28]:
def create_seasonal_naive_28_forecasts(
    data,
    folds,
    horizon=28,
):
    """
    Create forecasts using demand from exactly 28 days earlier.
    """

    history_lookup = (
        data[
            SERIES_KEYS
            + [
                "day_num",
                TARGET_COLUMN,
            ]
        ]
        .rename(
            columns={
                "day_num": "source_day_num",
                TARGET_COLUMN: "prediction",
            }
        )
    )

    forecast_frames = []

    for fold in folds.itertuples(index=False):
        validation_data = (
            data.loc[
                data["day_num"].between(
                    fold.validation_start_day,
                    fold.validation_end_day,
                ),
                SERIES_KEYS
                + [
                    "category_id",
                    "date",
                    "day_num",
                    TARGET_COLUMN,
                ],
            ]
            .copy()
        )

        validation_data["fold"] = fold.fold

        validation_data["horizon_day"] = (
            validation_data["day_num"]
            - fold.validation_start_day
            + 1
        )

        validation_data["source_day_num"] = (
            validation_data["day_num"]
            - horizon
        )

        validation_data["training_end_day"] = (
            fold.training_end_day
        )

        validation_data = validation_data.merge(
            history_lookup,
            on=SERIES_KEYS + ["source_day_num"],
            how="left",
            validate="many_to_one",
        )

        validation_data["model"] = (
            "Seasonal naive: lag 28"
        )

        forecast_frames.append(validation_data)

    forecasts = pd.concat(
        forecast_frames,
        ignore_index=True,
    )

    return forecasts

In [29]:
seasonal_naive_forecasts = (
    create_seasonal_naive_28_forecasts(
        data=df,
        folds=validation_folds,
        horizon=FORECAST_HORIZON,
    )
)

print(
    "Number of forecasts:",
    f"{len(seasonal_naive_forecasts):,}",
)

display(
    seasonal_naive_forecasts.head(10)
)

Number of forecasts: 13,160


,store_id,product_id,category_id,date,day_num,units_sold,fold,horizon_day,source_day_num,training_end_day,prediction,model
0,CA_1,FOODS_1_018,FOODS,2015-12-07,1774,8,1,1,1746,1773,5,Seasonal naive: lag 28
1,CA_1,FOODS_1_018,FOODS,2015-12-08,1775,8,1,2,1747,1773,2,Seasonal naive: lag 28
2,CA_1,FOODS_1_018,FOODS,2015-12-09,1776,7,1,3,1748,1773,4,Seasonal naive: lag 28
3,CA_1,FOODS_1_018,FOODS,2015-12-10,1777,2,1,4,1749,1773,2,Seasonal naive: lag 28
4,CA_1,FOODS_1_018,FOODS,2015-12-11,1778,2,1,5,1750,1773,2,Seasonal naive: lag 28
5,CA_1,FOODS_1_018,FOODS,2015-12-12,1779,5,1,6,1751,1773,8,Seasonal naive: lag 28
6,CA_1,FOODS_1_018,FOODS,2015-12-13,1780,6,1,7,1752,1773,6,Seasonal naive: lag 28
7,CA_1,FOODS_1_018,FOODS,2015-12-14,1781,6,1,8,1753,1773,4,Seasonal naive: lag 28
8,CA_1,FOODS_1_018,FOODS,2015-12-15,1782,5,1,9,1754,1773,3,Seasonal naive: lag 28
9,CA_1,FOODS_1_018,FOODS,2015-12-16,1783,10,1,10,1755,1773,3,Seasonal naive: lag 28


In [30]:
expected_forecast_rows = (
    N_VALIDATION_FOLDS
    * EXPECTED_NUMBER_OF_SERIES
    * FORECAST_HORIZON
)

assert (
    len(seasonal_naive_forecasts)
    == expected_forecast_rows
)

assert not seasonal_naive_forecasts[
    "prediction"
].isna().any()

assert seasonal_naive_forecasts[
    "horizon_day"
].between(
    1,
    FORECAST_HORIZON,
).all()

assert (
    seasonal_naive_forecasts["source_day_num"]
    <= seasonal_naive_forecasts["training_end_day"]
).all()

assert (
    seasonal_naive_forecasts["prediction"] >= 0
).all()

print("Seasonal-naïve forecast checks passed.")
print(
    "Latest source day used in each fold:"
)

display(
    seasonal_naive_forecasts
    .groupby("fold", as_index=False)
    .agg(
        training_end_day=(
            "training_end_day",
            "first",
        ),
        earliest_source_day=(
            "source_day_num",
            "min",
        ),
        latest_source_day=(
            "source_day_num",
            "max",
        ),
        validation_start_day=(
            "day_num",
            "min",
        ),
        validation_end_day=(
            "day_num",
            "max",
        ),
    )
)

Seasonal-naïve forecast checks passed.
Latest source day used in each fold:


,fold,training_end_day,earliest_source_day,latest_source_day,validation_start_day,validation_end_day
0,1,1773,1746,1773,1774,1801
1,2,1801,1774,1801,1802,1829
2,3,1829,1802,1829,1830,1857
3,4,1857,1830,1857,1858,1885
4,5,1885,1858,1885,1886,1913


## 4. Forecast Evaluation Metrics

The seasonal-naïve forecast is evaluated using four complementary metrics:

- **MAE:** Average absolute error in units. It is easy to interpret.
- **RMSE:** Penalizes large forecast errors more heavily than MAE.
- **WAPE:** Total absolute error as a percentage of total actual demand.
- **Bias:** Indicates whether forecasts systematically overestimate or
  underestimate demand.

A positive bias means overforecasting, while a negative bias means
underforecasting.

In [31]:
def calculate_forecast_metrics(
    forecast_data,
    actual_column=TARGET_COLUMN,
    prediction_column="prediction",
):
    actual = (
        forecast_data[actual_column]
        .to_numpy(dtype=float)
    )

    prediction = (
        forecast_data[prediction_column]
        .to_numpy(dtype=float)
    )

    error = prediction - actual
    absolute_error = np.abs(error)
    squared_error = np.square(error)

    total_actual = np.abs(actual).sum()

    mae = absolute_error.mean()
    rmse = np.sqrt(squared_error.mean())

    wape = (
        absolute_error.sum()
        / total_actual
        * 100
        if total_actual > 0
        else np.nan
    )

    bias = (
        error.sum()
        / total_actual
        * 100
        if total_actual > 0
        else np.nan
    )

    return {
        "number_of_forecasts": len(forecast_data),
        "actual_units": actual.sum(),
        "predicted_units": prediction.sum(),
        "mean_actual": actual.mean(),
        "mean_prediction": prediction.mean(),
        "mae": mae,
        "rmse": rmse,
        "wape_percentage": wape,
        "bias_percentage": bias,
    }

In [32]:
fold_metric_rows = []

for (
    model_name,
    fold_number,
), group in seasonal_naive_forecasts.groupby(
    [
        "model",
        "fold",
    ],
    sort=True,
):
    metrics = calculate_forecast_metrics(group)

    fold_metric_rows.append(
        {
            "model": model_name,
            "fold": fold_number,
            **metrics,
        }
    )

seasonal_naive_metrics_by_fold = pd.DataFrame(
    fold_metric_rows
)

display(
    seasonal_naive_metrics_by_fold.round(3)
)

,model,fold,number_of_forecasts,actual_units,predicted_units,mean_actual,mean_prediction,mae,rmse,wape_percentage,bias_percentage
0,Seasonal naive: lag 28,1,2632,"25,103.000","25,534.000",9.538,9.701,4.924,7.757,51.631,1.717
1,Seasonal naive: lag 28,2,2632,"23,089.000","25,103.000",8.772,9.538,5.135,7.974,58.539,8.723
2,Seasonal naive: lag 28,3,2632,"25,018.000","23,089.000",9.505,8.772,4.374,6.448,46.019,-7.710
3,Seasonal naive: lag 28,4,2632,"25,597.000","25,018.000",9.725,9.505,4.340,6.407,44.626,-2.262
4,Seasonal naive: lag 28,5,2632,"26,145.000","25,597.000",9.934,9.725,4.548,6.939,45.783,-2.096


In [33]:
overall_metrics = calculate_forecast_metrics(
    seasonal_naive_forecasts
)

seasonal_naive_overall_metrics = pd.DataFrame(
    [
        {
            "model": "Seasonal naive: lag 28",
            **overall_metrics,
        }
    ]
)

display(
    seasonal_naive_overall_metrics.round(3)
)

,model,number_of_forecasts,actual_units,predicted_units,mean_actual,mean_prediction,mae,rmse,wape_percentage,bias_percentage
0,Seasonal naive: lag 28,13160,"124,952.000","124,341.000",9.495,9.448,4.664,7.135,49.125,-0.489


In [35]:
metrics_by_fold = seasonal_naive_metrics_by_fold

seasonal_naive_fold_stability = pd.DataFrame(
    [
        {
            "number_of_folds": (
                metrics_by_fold["fold"].nunique()
            ),
            "mean_mae": (
                metrics_by_fold["mae"].mean()
            ),
            "standard_deviation_mae": (
                metrics_by_fold["mae"].std()
            ),
            "minimum_mae": (
                metrics_by_fold["mae"].min()
            ),
            "maximum_mae": (
                metrics_by_fold["mae"].max()
            ),
            "mean_rmse": (
                metrics_by_fold["rmse"].mean()
            ),
            "mean_wape_percentage": (
                metrics_by_fold[
                    "wape_percentage"
                ].mean()
            ),
            "standard_deviation_wape": (
                metrics_by_fold[
                    "wape_percentage"
                ].std()
            ),
            "minimum_wape_percentage": (
                metrics_by_fold[
                    "wape_percentage"
                ].min()
            ),
            "maximum_wape_percentage": (
                metrics_by_fold[
                    "wape_percentage"
                ].max()
            ),
            "mean_bias_percentage": (
                metrics_by_fold[
                    "bias_percentage"
                ].mean()
            ),
        }
    ]
)

display(
    seasonal_naive_fold_stability.round(3)
)

,number_of_folds,mean_mae,standard_deviation_mae,minimum_mae,maximum_mae,mean_rmse,mean_wape_percentage,standard_deviation_wape,minimum_wape_percentage,maximum_wape_percentage,mean_bias_percentage
0,5,4.664,0.351,4.340,5.135,7.105,49.320,5.826,44.626,58.539,-0.326


### Seasonal-Naïve Baseline Results

Across five validation folds, the 28-day seasonal-naïve model produced:

- **MAE:** 4.664 units
- **RMSE:** 7.135 units
- **WAPE:** 49.125%
- **Bias:** −0.489%

The near-zero overall bias indicates that total predicted demand is close to
total actual demand. The model predicted 124,341 units compared with 124,952
actual units, representing slight underforecasting.

However, the WAPE of approximately 49.1% shows that individual product-store
predictions still contain substantial errors. Correct aggregate demand does not
necessarily mean that demand was allocated correctly across products and days.

Performance also varies by validation period:

- Fold 4 performed best, with WAPE of 44.626%.
- Fold 2 performed worst, with WAPE of 58.539%.
- Fold bias ranged from −7.710% to +8.723%.

The difference between MAE and RMSE indicates that some observations have much
larger errors than the typical forecast. This is consistent with the
right-skewed demand distribution and occasional high-demand days found during
EDA.

This baseline now serves as the benchmark that subsequent forecasting methods
must improve upon.

## 5. Baseline 2: Repeated Last-Week Pattern

This baseline takes the final seven observed days before each forecast origin
and repeats them four times to produce the 28-day forecast.

For example:

- Forecast horizon day 1 uses the corresponding day from the previous week.
- Forecast horizon day 8 repeats the prediction from horizon day 1.
- The same seven-day pattern is repeated throughout the forecast horizon.

This approach is leakage-safe because it only uses the seven days available
before forecasting begins. It tests whether very recent weekly demand is more
informative than demand observed exactly four weeks earlier.

In [36]:
def create_repeated_last_week_forecasts(
    data,
    folds,
):
    """
    Repeat the final seven observed training days across
    the complete 28-day forecast horizon.
    """

    history_lookup = (
        data[
            SERIES_KEYS
            + [
                "day_num",
                TARGET_COLUMN,
            ]
        ]
        .rename(
            columns={
                "day_num": "source_day_num",
                TARGET_COLUMN: "prediction",
            }
        )
    )

    forecast_frames = []

    for fold in folds.itertuples(index=False):
        validation_data = (
            data.loc[
                data["day_num"].between(
                    fold.validation_start_day,
                    fold.validation_end_day,
                ),
                SERIES_KEYS
                + [
                    "category_id",
                    "date",
                    "day_num",
                    TARGET_COLUMN,
                ],
            ]
            .copy()
        )

        validation_data["fold"] = fold.fold

        validation_data["horizon_day"] = (
            validation_data["day_num"]
            - fold.validation_start_day
            + 1
        )

        # The last seven training days run from
        # training_end_day - 6 through training_end_day.
        validation_data["source_day_num"] = (
            fold.training_end_day
            - 6
            + (
                (
                    validation_data["horizon_day"]
                    - 1
                )
                % 7
            )
        )

        validation_data["training_end_day"] = (
            fold.training_end_day
        )

        validation_data = validation_data.merge(
            history_lookup,
            on=SERIES_KEYS + ["source_day_num"],
            how="left",
            validate="many_to_one",
        )

        validation_data["model"] = (
            "Repeated last-week pattern"
        )

        forecast_frames.append(validation_data)

    return pd.concat(
        forecast_frames,
        ignore_index=True,
    )

In [37]:
repeated_week_forecasts = (
    create_repeated_last_week_forecasts(
        data=df,
        folds=validation_folds,
    )
)

assert (
    len(repeated_week_forecasts)
    == expected_forecast_rows
)

assert not repeated_week_forecasts[
    "prediction"
].isna().any()

assert repeated_week_forecasts[
    "horizon_day"
].between(
    1,
    FORECAST_HORIZON,
).all()

assert (
    repeated_week_forecasts["source_day_num"]
    <= repeated_week_forecasts["training_end_day"]
).all()

assert (
    repeated_week_forecasts["source_day_num"]
    >= repeated_week_forecasts["training_end_day"] - 6
).all()

assert (
    repeated_week_forecasts["prediction"] >= 0
).all()

print("Repeated-week forecast checks passed.")
print(
    "Number of forecasts:",
    f"{len(repeated_week_forecasts):,}",
)

Repeated-week forecast checks passed.
Number of forecasts: 13,160


In [38]:
display(
    repeated_week_forecasts.loc[
        (
            repeated_week_forecasts["fold"] == 1
        )
        & (
            repeated_week_forecasts["store_id"] == "CA_1"
        )
        & (
            repeated_week_forecasts["product_id"]
            == selected_products[0]
        ),
        [
            "fold",
            "store_id",
            "product_id",
            "horizon_day",
            "source_day_num",
            "day_num",
            TARGET_COLUMN,
            "prediction",
        ],
    ]
)

,fold,store_id,product_id,horizon_day,source_day_num,day_num,units_sold,prediction
616,1,CA_1,FOODS_3_555,1,1767,1774,18,9
617,1,CA_1,FOODS_3_555,2,1768,1775,8,9
618,1,CA_1,FOODS_3_555,3,1769,1776,17,13
619,1,CA_1,FOODS_3_555,4,1770,1777,10,16
620,1,CA_1,FOODS_3_555,5,1771,1778,20,23
621,1,CA_1,FOODS_3_555,6,1772,1779,27,31
622,1,CA_1,FOODS_3_555,7,1773,1780,26,26
623,1,CA_1,FOODS_3_555,8,1767,1781,17,9
624,1,CA_1,FOODS_3_555,9,1768,1782,14,9
625,1,CA_1,FOODS_3_555,10,1769,1783,12,13


In [39]:
repeated_week_fold_metric_rows = []

for (
    model_name,
    fold_number,
), group in repeated_week_forecasts.groupby(
    [
        "model",
        "fold",
    ],
    sort=True,
):
    metrics = calculate_forecast_metrics(group)

    repeated_week_fold_metric_rows.append(
        {
            "model": model_name,
            "fold": fold_number,
            **metrics,
        }
    )

repeated_week_metrics_by_fold = pd.DataFrame(
    repeated_week_fold_metric_rows
)

display(
    repeated_week_metrics_by_fold.round(3)
)

,model,fold,number_of_forecasts,actual_units,predicted_units,mean_actual,mean_prediction,mae,rmse,wape_percentage,bias_percentage
0,Repeated last-week pattern,1,2632,"25,103.000","26,460.000",9.538,10.053,5.136,7.738,53.854,5.406
1,Repeated last-week pattern,2,2632,"23,089.000","23,636.000",8.772,8.980,4.260,6.179,48.564,2.369
2,Repeated last-week pattern,3,2632,"25,018.000","23,336.000",9.505,8.866,4.388,6.506,46.159,-6.723
3,Repeated last-week pattern,4,2632,"25,597.000","24,032.000",9.725,9.131,4.308,6.464,44.298,-6.114
4,Repeated last-week pattern,5,2632,"26,145.000","26,532.000",9.934,10.081,4.463,6.846,44.930,1.480


In [40]:
repeated_week_overall_metrics = pd.DataFrame(
    [
        {
            "model": "Repeated last-week pattern",
            **calculate_forecast_metrics(
                repeated_week_forecasts
            ),
        }
    ]
)

display(
    repeated_week_overall_metrics.round(3)
)

,model,number_of_forecasts,actual_units,predicted_units,mean_actual,mean_prediction,mae,rmse,wape_percentage,bias_percentage
0,Repeated last-week pattern,13160,"124,952.000","123,996.000",9.495,9.422,4.511,6.768,47.511,-0.765


In [41]:
baseline_overall_comparison = pd.concat(
    [
        seasonal_naive_overall_metrics,
        repeated_week_overall_metrics,
    ],
    ignore_index=True,
)

baseline_overall_comparison = (
    baseline_overall_comparison
    .sort_values("wape_percentage")
    .reset_index(drop=True)
)

display(
    baseline_overall_comparison[
        [
            "model",
            "number_of_forecasts",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ]
    ]
    .round(3)
)

,model,number_of_forecasts,mae,rmse,wape_percentage,bias_percentage
0,Repeated last-week pattern,13160,4.511,6.768,47.511,-0.765
1,Seasonal naive: lag 28,13160,4.664,7.135,49.125,-0.489


### Repeated Last-Week Baseline Results

Repeating the most recent seven-day demand pattern outperformed the 28-day
seasonal-naïve forecast across all three accuracy metrics.

| Metric | Lag-28 baseline | Repeated-week baseline | Result |
|---|---:|---:|---|
| MAE | 4.664 | 4.511 | 3.3% improvement |
| RMSE | 7.135 | 6.768 | 5.1% improvement |
| WAPE | 49.125% | 47.511% | 1.614 percentage-point improvement |
| Bias | −0.489% | −0.765% | Slightly more underforecasting |

The result suggests that the most recent weekly demand level contains more
useful forecasting information than demand from exactly four weeks earlier.

The lower RMSE is particularly encouraging because it indicates fewer or
smaller large forecast errors. However, the repeated-week baseline may be
sensitive to an unusually high or low final training week.

The repeated last-week pattern is therefore the current benchmark, with a WAPE
of 47.511%. Subsequent methods must improve upon this result.

## 6. Baseline 3: Recent Historical Mean

This baseline calculates the mean demand during the final observed days before
each forecast origin.

Three history windows are tested:

- 7-day mean
- 14-day mean
- 28-day mean

The calculated mean becomes a constant prediction for all 28 forecast days for
that product-store series.

This approach smooths short-term demand fluctuations and reduces sensitivity to
an unusually high or low individual day. However, it does not directly preserve
day-of-week patterns.

In [42]:
def create_recent_mean_forecasts(
    data,
    folds,
    windows=(7, 14, 28),
):
    """
    Create constant forecasts from the mean demand during
    the final observed days before each forecast origin.
    """

    forecast_frames = []

    for fold in folds.itertuples(index=False):
        validation_template = (
            data.loc[
                data["day_num"].between(
                    fold.validation_start_day,
                    fold.validation_end_day,
                ),
                SERIES_KEYS
                + [
                    "category_id",
                    "date",
                    "day_num",
                    TARGET_COLUMN,
                ],
            ]
            .copy()
        )

        validation_template["fold"] = fold.fold

        validation_template["horizon_day"] = (
            validation_template["day_num"]
            - fold.validation_start_day
            + 1
        )

        validation_template["training_end_day"] = (
            fold.training_end_day
        )

        for window in windows:
            history_start_day = (
                fold.training_end_day
                - window
                + 1
            )

            recent_history = data.loc[
                data["day_num"].between(
                    history_start_day,
                    fold.training_end_day,
                )
            ]

            recent_means = (
                recent_history
                .groupby(
                    SERIES_KEYS,
                    as_index=False,
                )
                .agg(
                    prediction=(
                        TARGET_COLUMN,
                        "mean",
                    )
                )
            )

            forecasts = validation_template.merge(
                recent_means,
                on=SERIES_KEYS,
                how="left",
                validate="many_to_one",
            )

            forecasts["source_start_day"] = (
                history_start_day
            )

            forecasts["source_end_day"] = (
                fold.training_end_day
            )

            forecasts["window"] = window

            forecasts["model"] = (
                f"Recent {window}-day mean"
            )

            forecast_frames.append(forecasts)

    return pd.concat(
        forecast_frames,
        ignore_index=True,
    )

In [43]:
RECENT_MEAN_WINDOWS = [
    7,
    14,
    28,
]

recent_mean_forecasts = (
    create_recent_mean_forecasts(
        data=df,
        folds=validation_folds,
        windows=RECENT_MEAN_WINDOWS,
    )
)

print(
    "Number of forecasts:",
    f"{len(recent_mean_forecasts):,}",
)

Number of forecasts: 39,480


In [44]:
expected_recent_mean_rows = (
    len(RECENT_MEAN_WINDOWS)
    * expected_forecast_rows
)

assert (
    len(recent_mean_forecasts)
    == expected_recent_mean_rows
)

assert not recent_mean_forecasts[
    "prediction"
].isna().any()

assert (
    recent_mean_forecasts["prediction"] >= 0
).all()

assert (
    recent_mean_forecasts["source_end_day"]
    <= recent_mean_forecasts["training_end_day"]
).all()

assert (
    recent_mean_forecasts["source_start_day"]
    <= recent_mean_forecasts["source_end_day"]
).all()

forecasts_per_model = (
    recent_mean_forecasts
    .groupby("model")
    .size()
)

assert (
    forecasts_per_model
    == expected_forecast_rows
).all()

print("Recent-mean forecast checks passed.")

display(
    forecasts_per_model
    .rename("number_of_forecasts")
    .reset_index()
)

Recent-mean forecast checks passed.


,model,number_of_forecasts
0,Recent 14-day mean,13160
1,Recent 28-day mean,13160
2,Recent 7-day mean,13160


In [45]:
recent_mean_fold_metric_rows = []

for (
    model_name,
    fold_number,
), group in recent_mean_forecasts.groupby(
    [
        "model",
        "fold",
    ],
    sort=True,
):
    metrics = calculate_forecast_metrics(group)

    recent_mean_fold_metric_rows.append(
        {
            "model": model_name,
            "fold": fold_number,
            **metrics,
        }
    )

recent_mean_metrics_by_fold = pd.DataFrame(
    recent_mean_fold_metric_rows
)

display(
    recent_mean_metrics_by_fold[
        [
            "model",
            "fold",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ]
    ]
    .round(3)
)

,model,fold,mae,rmse,wape_percentage,bias_percentage
0,Recent 14-day mean,1,4.067,6.309,42.637,3.772
1,Recent 14-day mean,2,3.791,5.624,43.214,3.677
2,Recent 14-day mean,3,3.621,5.534,38.090,-8.410
3,Recent 14-day mean,4,3.653,5.694,37.560,-3.153
4,Recent 14-day mean,5,3.643,5.446,36.673,-2.689
5,Recent 28-day mean,1,4.013,6.245,42.077,1.717
6,Recent 28-day mean,2,3.975,6.007,45.315,8.723
7,Recent 28-day mean,3,3.662,5.574,38.525,-7.710
8,Recent 28-day mean,4,3.577,5.613,36.779,-2.262
9,Recent 28-day mean,5,3.686,5.488,37.108,-2.096


In [46]:
recent_mean_overall_rows = []

for model_name, group in recent_mean_forecasts.groupby(
    "model",
    sort=True,
):
    recent_mean_overall_rows.append(
        {
            "model": model_name,
            **calculate_forecast_metrics(group),
        }
    )

recent_mean_overall_metrics = pd.DataFrame(
    recent_mean_overall_rows
)

In [47]:
all_baseline_overall_comparison = pd.concat(
    [
        seasonal_naive_overall_metrics,
        repeated_week_overall_metrics,
        recent_mean_overall_metrics,
    ],
    ignore_index=True,
)

all_baseline_overall_comparison = (
    all_baseline_overall_comparison
    .sort_values("wape_percentage")
    .reset_index(drop=True)
)

display(
    all_baseline_overall_comparison[
        [
            "model",
            "number_of_forecasts",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ]
    ]
    .round(3)
)

,model,number_of_forecasts,mae,rmse,wape_percentage,bias_percentage
0,Recent 14-day mean,13160,3.755,5.729,39.545,-1.455
1,Recent 28-day mean,13160,3.783,5.793,39.839,-0.489
2,Recent 7-day mean,13160,3.837,5.815,40.414,-0.765
3,Repeated last-week pattern,13160,4.511,6.768,47.511,-0.765
4,Seasonal naive: lag 28,13160,4.664,7.135,49.125,-0.489


### Recent-Mean Baseline Results

All three recent-mean baselines substantially outperformed the seasonal-pattern
baselines.

The 14-day mean achieved the strongest overall accuracy:

- **MAE:** 3.755 units
- **RMSE:** 5.729 units
- **WAPE:** 39.545%
- **Bias:** −1.455%

Compared with the previous best model, the repeated last-week pattern, the
14-day mean reduced MAE by approximately 16.8% and RMSE by approximately 15.4%.
Its WAPE improved from 47.511% to 39.545%.

The 7-, 14-, and 28-day results are relatively close, but the 14-day window
provides the best balance between responsiveness and smoothing:

- The 7-day mean may react too strongly to an unusual recent week.
- The 28-day mean is more stable but may respond too slowly to recent changes.
- The 14-day mean smooths daily noise while remaining responsive to current
  demand levels.

The 28-day mean has the smallest bias among the recent-mean models, but its
point-level accuracy is slightly weaker than the 14-day mean.

The strong performance of constant mean forecasts indicates that smoothing
product-level demand is currently more valuable than directly repeating daily
patterns. Nevertheless, calendar and weekday features may allow a machine-
learning model to recover systematic weekend effects without repeating noisy
individual observations.

The recent 14-day mean is now the leading baseline, with a WAPE of 39.545%.

## 7. Baseline Stability Across Validation Folds

Overall performance can hide periods in which a forecasting method performs
poorly. Therefore, the five baselines are compared separately within every
validation fold.

A reliable benchmark should:

- achieve low average error;
- perform consistently across time;
- avoid unusually poor validation periods;
- win across multiple folds rather than only one period.

In [48]:
all_baseline_metrics_by_fold = pd.concat(
    [
        seasonal_naive_metrics_by_fold,
        repeated_week_metrics_by_fold,
        recent_mean_metrics_by_fold,
    ],
    ignore_index=True,
)

assert (
    all_baseline_metrics_by_fold
    .groupby("model")["fold"]
    .nunique()
    .eq(N_VALIDATION_FOLDS)
    .all()
)

display(
    all_baseline_metrics_by_fold[
        [
            "model",
            "fold",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ]
    ]
    .sort_values(
        [
            "fold",
            "wape_percentage",
        ]
    )
    .round(3)
)

,model,fold,mae,rmse,wape_percentage,bias_percentage
15,Recent 28-day mean,1,4.013,6.245,42.077,1.717
10,Recent 14-day mean,1,4.067,6.309,42.637,3.772
20,Recent 7-day mean,1,4.235,6.402,44.398,5.406
0,Seasonal naive: lag 28,1,4.924,7.757,51.631,1.717
5,Repeated last-week pattern,1,5.136,7.738,53.854,5.406
21,Recent 7-day mean,2,3.643,5.345,41.531,2.369
11,Recent 14-day mean,2,3.791,5.624,43.214,3.677
16,Recent 28-day mean,2,3.975,6.007,45.315,8.723
6,Repeated last-week pattern,2,4.260,6.179,48.564,2.369
1,Seasonal naive: lag 28,2,5.135,7.974,58.539,8.723


In [49]:
wape_by_fold = (
    all_baseline_metrics_by_fold
    .pivot(
        index="fold",
        columns="model",
        values="wape_percentage",
    )
)

display(
    wape_by_fold.round(3)
)

model,Recent 14-day mean,Recent 28-day mean,Recent 7-day mean,Repeated last-week pattern,Seasonal naive: lag 28
fold,,,,,
1,42.637,42.077,44.398,53.854,51.631
2,43.214,45.315,41.531,48.564,58.539
3,38.090,38.525,40.391,46.159,46.019
4,37.560,36.779,38.085,44.298,44.626
5,36.673,37.108,37.905,44.930,45.783


In [50]:
best_row_indices = (
    all_baseline_metrics_by_fold
    .groupby("fold")["wape_percentage"]
    .idxmin()
)

best_model_by_fold = (
    all_baseline_metrics_by_fold
    .loc[
        best_row_indices,
        [
            "fold",
            "model",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ],
    ]
    .sort_values("fold")
    .reset_index(drop=True)
)

display(
    best_model_by_fold.round(3)
)

,fold,model,mae,rmse,wape_percentage,bias_percentage
0,1,Recent 28-day mean,4.013,6.245,42.077,1.717
1,2,Recent 7-day mean,3.643,5.345,41.531,2.369
2,3,Recent 14-day mean,3.621,5.534,38.090,-8.410
3,4,Recent 28-day mean,3.577,5.613,36.779,-2.262
4,5,Recent 14-day mean,3.643,5.446,36.673,-2.689


In [51]:
fold_win_counts = (
    best_model_by_fold["model"]
    .value_counts()
    .rename_axis("model")
    .reset_index(name="number_of_fold_wins")
)

model_stability_summary = (
    all_baseline_metrics_by_fold
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        mean_mae=("mae", "mean"),
        standard_deviation_mae=("mae", "std"),
        mean_rmse=("rmse", "mean"),
        mean_wape_percentage=(
            "wape_percentage",
            "mean",
        ),
        standard_deviation_wape=(
            "wape_percentage",
            "std",
        ),
        minimum_wape_percentage=(
            "wape_percentage",
            "min",
        ),
        maximum_wape_percentage=(
            "wape_percentage",
            "max",
        ),
        mean_bias_percentage=(
            "bias_percentage",
            "mean",
        ),
    )
    .merge(
        fold_win_counts,
        on="model",
        how="left",
    )
)

model_stability_summary[
    "number_of_fold_wins"
] = (
    model_stability_summary[
        "number_of_fold_wins"
    ]
    .fillna(0)
    .astype(int)
)

model_stability_summary = (
    model_stability_summary
    .sort_values("mean_wape_percentage")
    .reset_index(drop=True)
)

display(
    model_stability_summary.round(3)
)

,model,mean_mae,standard_deviation_mae,mean_rmse,mean_wape_percentage,standard_deviation_wape,minimum_wape_percentage,maximum_wape_percentage,mean_bias_percentage,number_of_fold_wins
0,Recent 14-day mean,3.755,0.187,5.721,39.635,3.053,36.673,43.214,-1.360,2
1,Recent 28-day mean,3.783,0.198,5.786,39.961,3.656,36.779,45.315,-0.326,2
2,Recent 7-day mean,3.837,0.234,5.804,40.462,2.685,37.905,44.398,-0.716,1
3,Repeated last-week pattern,4.511,0.358,6.747,47.561,3.878,44.298,53.854,-0.716,0
4,Seasonal naive: lag 28,4.664,0.351,7.105,49.320,5.826,44.626,58.539,-0.326,0


### Baseline Stability Findings

The recent 14-day mean remains the strongest overall benchmark.

It achieved:

- Mean MAE of 3.755 units
- Mean RMSE of 5.721 units
- Mean fold WAPE of 39.635%
- WAPE between 36.673% and 43.214%
- Two wins across the five validation folds

The 28-day mean also won two folds, but its average WAPE was slightly higher at
39.961%. Its worst-fold WAPE of 45.315% was also higher than the 14-day model's
worst result of 43.214%.

The 7-day mean won one fold and had the lowest standard deviation of WAPE.
However, its average errors were higher, suggesting that it is stable but
slightly less accurate.

The seasonal-pattern baselines did not win any validation fold. This confirms
that smoothing recent product-level demand is more effective than directly
copying historical daily observations.

Although the 14-day mean only won two folds, it achieved:

- the lowest average MAE;
- the lowest average RMSE;
- the lowest average WAPE;
- the lowest worst-fold WAPE.

Therefore, the recent 14-day mean is selected as the primary baseline. The
recent 28-day mean will remain a useful secondary benchmark because it has
smaller forecast bias.

## 8. Baseline Performance by Store and Category

Overall metrics can hide important differences between business segments.

A model may perform well for high-volume Foods products but poorly for
lower-volume or intermittent Hobbies products. Performance is therefore
evaluated separately by:

- store;
- product category.

The analysis uses all five rolling-origin validation folds combined. WAPE is
used as the main comparison metric, while MAE, RMSE, and bias provide additional
context.

In [52]:
all_baseline_forecasts = pd.concat(
    [
        seasonal_naive_forecasts,
        repeated_week_forecasts,
        recent_mean_forecasts,
    ],
    ignore_index=True,
    sort=False,
)

EXPECTED_NUMBER_OF_MODELS = 5

expected_all_baseline_rows = (
    EXPECTED_NUMBER_OF_MODELS
    * expected_forecast_rows
)

assert (
    len(all_baseline_forecasts)
    == expected_all_baseline_rows
)

assert (
    all_baseline_forecasts["model"].nunique()
    == EXPECTED_NUMBER_OF_MODELS
)

assert not all_baseline_forecasts[
    "prediction"
].isna().any()

print("Combined baseline forecast checks passed.")
print(
    "Total forecast rows:",
    f"{len(all_baseline_forecasts):,}",
)

display(
    all_baseline_forecasts[
        "model"
    ]
    .value_counts()
    .rename_axis("model")
    .reset_index(name="number_of_forecasts")
)

Combined baseline forecast checks passed.
Total forecast rows: 65,800


,model,number_of_forecasts
0,Seasonal naive: lag 28,13160
1,Repeated last-week pattern,13160
2,Recent 7-day mean,13160
3,Recent 14-day mean,13160
4,Recent 28-day mean,13160


In [53]:
def calculate_grouped_forecast_metrics(
    forecast_data,
    group_columns,
):
    metric_rows = []

    grouped_data = forecast_data.groupby(
        group_columns,
        sort=True,
        dropna=False,
    )

    for group_values, group in grouped_data:
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        group_identifiers = dict(
            zip(
                group_columns,
                group_values,
            )
        )

        metric_rows.append(
            {
                **group_identifiers,
                **calculate_forecast_metrics(group),
            }
        )

    return pd.DataFrame(metric_rows)

In [54]:
baseline_metrics_by_store = (
    calculate_grouped_forecast_metrics(
        forecast_data=all_baseline_forecasts,
        group_columns=[
            "store_id",
            "model",
        ],
    )
)

display(
    baseline_metrics_by_store[
        [
            "store_id",
            "model",
            "number_of_forecasts",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ]
    ]
    .sort_values(
        [
            "store_id",
            "wape_percentage",
        ]
    )
    .round(3)
)

,store_id,model,number_of_forecasts,mae,rmse,wape_percentage,bias_percentage
1,CA_1,Recent 28-day mean,6580,3.504,5.375,44.501,0.295
0,CA_1,Recent 14-day mean,6580,3.521,5.374,44.723,0.757
2,CA_1,Recent 7-day mean,6580,3.588,5.475,45.573,0.606
3,CA_1,Repeated last-week pattern,6580,4.070,6.216,51.697,0.606
4,CA_1,Seasonal naive: lag 28,6580,4.128,6.389,52.436,0.295
5,CA_3,Recent 14-day mean,6580,3.988,6.064,35.878,-3.021
6,CA_3,Recent 28-day mean,6580,4.062,6.182,36.537,-1.044
7,CA_3,Recent 7-day mean,6580,4.086,6.136,36.761,-1.736
8,CA_3,Repeated last-week pattern,6580,4.952,7.279,44.547,-1.736
9,CA_3,Seasonal naive: lag 28,6580,5.200,7.809,46.780,-1.044


In [56]:
store_wape_comparison = (
    baseline_metrics_by_store
    .pivot(
        index="store_id",
        columns="model",
        values="wape_percentage",
    )
)

display(
    store_wape_comparison.round(3)
)

model,Recent 14-day mean,Recent 28-day mean,Recent 7-day mean,Repeated last-week pattern,Seasonal naive: lag 28
store_id,,,,,
CA_1,44.723,44.501,45.573,51.697,52.436
CA_3,35.878,36.537,36.761,44.547,46.780


In [57]:
best_store_row_indices = (
    baseline_metrics_by_store
    .groupby("store_id")["wape_percentage"]
    .idxmin()
)

best_model_by_store = (
    baseline_metrics_by_store
    .loc[
        best_store_row_indices,
        [
            "store_id",
            "model",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ],
    ]
    .sort_values("store_id")
    .reset_index(drop=True)
)

display(
    best_model_by_store.round(3)
)

,store_id,model,mae,rmse,wape_percentage,bias_percentage
0,CA_1,Recent 28-day mean,3.504,5.375,44.501,0.295
1,CA_3,Recent 14-day mean,3.988,6.064,35.878,-3.021


In [58]:
baseline_metrics_by_category = (
    calculate_grouped_forecast_metrics(
        forecast_data=all_baseline_forecasts,
        group_columns=[
            "category_id",
            "model",
        ],
    )
)

display(
    baseline_metrics_by_category[
        [
            "category_id",
            "model",
            "number_of_forecasts",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ]
    ]
    .sort_values(
        [
            "category_id",
            "wape_percentage",
        ]
    )
    .round(3)
)

,category_id,model,number_of_forecasts,mae,rmse,wape_percentage,bias_percentage
0,FOODS,Recent 14-day mean,8400,4.118,6.116,34.700,-1.177
1,FOODS,Recent 28-day mean,8400,4.171,6.237,35.147,-0.326
2,FOODS,Recent 7-day mean,8400,4.183,6.153,35.250,-1.321
3,FOODS,Repeated last-week pattern,8400,4.855,6.981,40.908,-1.321
4,FOODS,Seasonal naive: lag 28,8400,5.030,7.447,42.385,-0.326
6,HOBBIES,Recent 28-day mean,1400,4.188,6.673,70.628,0.867
5,HOBBIES,Recent 14-day mean,1400,4.199,6.703,70.806,0.843
7,HOBBIES,Recent 7-day mean,1400,4.338,6.830,73.153,6.191
8,HOBBIES,Repeated last-week pattern,1400,5.347,8.629,90.171,6.191
9,HOBBIES,Seasonal naive: lag 28,1400,5.586,9.174,94.194,0.867


In [59]:
category_wape_comparison = (
    baseline_metrics_by_category
    .pivot(
        index="category_id",
        columns="model",
        values="wape_percentage",
    )
)

display(
    category_wape_comparison.round(3)
)

model,Recent 14-day mean,Recent 28-day mean,Recent 7-day mean,Repeated last-week pattern,Seasonal naive: lag 28
category_id,,,,,
FOODS,34.700,35.147,35.250,40.908,42.385
HOBBIES,70.806,70.628,73.153,90.171,94.194
HOUSEHOLD,52.723,52.347,54.740,65.438,66.682


In [60]:
best_category_row_indices = (
    baseline_metrics_by_category
    .groupby("category_id")[
        "wape_percentage"
    ]
    .idxmin()
)

best_model_by_category = (
    baseline_metrics_by_category
    .loc[
        best_category_row_indices,
        [
            "category_id",
            "model",
            "mae",
            "rmse",
            "wape_percentage",
            "bias_percentage",
        ],
    ]
    .sort_values("category_id")
    .reset_index(drop=True)
)

display(
    best_model_by_category.round(3)
)

,category_id,model,mae,rmse,wape_percentage,bias_percentage
0,FOODS,Recent 14-day mean,4.118,6.116,34.700,-1.177
1,HOBBIES,Recent 28-day mean,4.188,6.673,70.628,0.867
2,HOUSEHOLD,Recent 28-day mean,2.642,3.954,52.347,-2.111


In [ ]:
### Store and Category Findings

Baseline performance differs meaningfully across stores and product categories.

#### Store-level results

For CA_1, the recent 28-day mean performed best:

- MAE: 3.504 units
- RMSE: 5.375 units
- WAPE: 44.501%
- Bias: +0.295%

For CA_3, the recent 14-day mean performed best:

- MAE: 3.988 units
- RMSE: 6.064 units
- WAPE: 35.878%
- Bias: −3.021%

CA_3 has a higher MAE but a lower WAPE because its overall sales volume is
higher. Therefore, an error of approximately four units represents a smaller
percentage of CA_3 demand than it does for CA_1.

The longer 28-day window appears more suitable for CA_1, while CA_3 benefits
from a more responsive 14-day window.

#### Category-level results

Foods achieved the strongest relative forecast accuracy, with the recent
14-day mean producing a WAPE of 34.700%.

Hobbies was the most difficult category to forecast, with a best WAPE of
70.628%. Although its MAE was similar to Foods, Hobbies has substantially lower
sales volume. Consequently, a similar absolute error produces a much larger
percentage error.

Household achieved a best WAPE of 52.347% using the recent 28-day mean.

The results support the EDA findings:

- Foods demand benefits from a shorter and more responsive history window.
- Hobbies and Household benefit from longer averaging that reduces noise.
- Lower-volume and more irregular series are more difficult to forecast
  accurately in percentage terms.

For a consistent project-wide benchmark, the recent 14-day mean remains the
primary baseline. However, the segment results indicate that a future model
should learn different behaviours across stores and categories rather than
applying one identical demand rule to every series.